# Inspection report

[`01-tolerance-tightness`](01-tolerance-tightness.ipynb) shows **design intent** -
how tightly each face is controlled. From here on the notebooks show **what the shop
actually made**, read out of the results section of the same kind of QIF document.

This one builds the table every later notebook starts from, sets aside the rows that
should not be trusted, and shows one part the way an inspector would see it. No 3D.

Setup is in the [README](../README.md).

## Setup


`%autoreload 2` re-reads [`qifkit.py`](qifkit.py) before every cell, so an edit
to it takes effect without restarting the kernel.

The helpers are in [`qifkit.py`](qifkit.py), section 3. How a measurement is joined
to its feature is walked through by hand in [`00-reading-qif`](00-reading-qif.ipynb).

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

import qifkit

pd.set_option('display.width', 140)

from qifkit import NS

## The document

The `-MV-FH` files are complete QIF documents: B-rep geometry, PMI **and**
results together, with their own `QPId`. Nothing has to be joined across files.

In [2]:
MODEL = '000454_D-MV-FH-2'
doc = qifkit.load(MODEL)

print(f'{MODEL}: {len(doc.by_id)} identified elements, '
      f'{len(doc.root.findall(".//qif:MeasurementResults", NS))} measurement results')

000454_D-MV-FH-2: 4626 identified elements, 15 measurement results


## Building the table

One row per `CharacteristicMeasurement`, which is one **application** of a callout
on one part:

- a callout on the drawing is a `CharacteristicNominal`; Ø8.73 applied to four holes
  is one nominal
- each place it applies is a `CharacteristicItem`; four holes, four items
- each measured part has one measurement per item

So that callout contributes **4 rows per part**, 60 over 15 parts. An application
that spans more than one feature (a distance *between* two faces) is still one
row, and carries both faces in `entities`.

The join from measurement to geometry goes through the `CharacteristicItem`, not
the callout, so each hole's row carries only its own face. See the diagram in
[`00-reading-qif`](00-reading-qif.ipynb). Parts are told apart by serial number,
which the two sample files record in two different places; `serial_number` reads
both.

`fraction` is how much of the tolerance band the measurement consumed: 1.0 sits
exactly on a limit, beyond ±1.0 is out of tolerance. It is what makes
characteristics with different nominals and tolerances comparable on one axis.

In [3]:
measurements = qifkit.measurement_table(doc)

print(f'{len(measurements)} rows, {measurements.serial.nunique()} parts, '
      f'{measurements.characteristic.nunique()} characteristics')
measurements.head(8)

600 rows, 15 parts, 41 characteristics


,serial,characteristic,name,type,nominal,low,high,value,status_from_file,entities,fraction
0,SN#1,Flatness Item 5715,Feature Control Frame (180),Flatness,NaN,0.00,0.25,0.010372,PASS,"(877,)",0.041490
1,SN#1,Diameter Item 5673,Radial Dimension (192),Diameter,22.10,-0.05,0.20,22.098708,PASS,"(1174,)",-0.025834
2,SN#1,Perpendicularity Item 5717,Feature Control Frame (196),Perpendicularity,NaN,0.00,0.50,22.087000,PASS,"(1174,)",44.174000
3,SN#1,Diameter Item 5675,Radial Dimension (191),Diameter,14.05,-0.05,0.20,14.030424,PASS,"(1144,)",-0.391524
4,SN#1,Perpendicularity Item 5719,Feature Control Frame (195),Perpendicularity,NaN,0.00,0.50,0.009387,PASS,"(1144,)",0.018774
5,SN#1,Distance Between Item 5693,Linear Dimension (114),DistanceBetween,32.00,-0.13,0.13,32.007932,PASS,"(120, 877)",0.061015
6,SN#1,Distance Between Item 5696,Linear Dimension (114),DistanceBetween,32.00,-0.13,0.13,31.991632,PASS,"(414, 877)",-0.064366
7,SN#1,Parallelism Item 5743,Feature Control Frame (146),Parallelism,NaN,0.00,0.50,0.012918,PASS,"(120,)",0.025836


## Rows to set aside

A file straight off a CMM is not guaranteed to be tidy. Before charting anything,
three kinds of row are set aside, and the cell **prints how many of each**, rather
than quietly deleting anything that looks incomplete.

1. **Reported, never measured:** listed for the part, but with no value.
2. **No tolerance:** nothing to judge the value against.
3. **Suspect export:** a geometric tolerance reading more than 20 times its own
   zone. Here one perpendicularity reads **22.087 mm against a 0.5 mm zone**: the
   export wrote the feature's diameter into it. That is a data error, not a bad part.

The rules are in `qifkit.set_aside`. It returns each kind separately rather than
one filtered table, so this cell can say how many of each went and why. The quiet
alternative, pandas' `dropna()`, would drop them in one call and hide the reason.

These rows are specific to this document. Switch `MODEL` to `000473_A-MV-FH` and all
three counts come back zero. A clean file is a normal outcome too, which is why it
is checked rather than assumed.

In [4]:
parts = qifkit.set_aside(measurements)
unmeasured, untoleranced, suspect = parts['unmeasured'], parts['untoleranced'], parts['suspect']

print(f'{len(unmeasured):4} rows with no Value         (reported, never measured)')
print(f'{len(untoleranced):4} rows with no tolerance    (nothing to compare against)')
print(f'{len(suspect):4} rows over {qifkit.SUSPECT_RATIO}x their zone (suspect export, shown below)')

if len(suspect):
    display(suspect[['serial', 'name', 'type', 'value', 'high', 'fraction']].head(3))

clean = parts['clean']
print(f'\n{len(clean)} rows kept, {clean.serial.nunique()} parts, '
      f'{clean.characteristic.nunique()} characteristics')

 120 rows with no Value         (reported, never measured)
   0 rows with no tolerance    (nothing to compare against)
  15 rows over 20x their zone (suspect export, shown below)


,serial,name,type,value,high,fraction
2,SN#1,Feature Control Frame (196),Perpendicularity,22.087,0.5,44.174
42,SN#2,Feature Control Frame (196),Perpendicularity,22.084,0.5,44.168
82,SN#3,Feature Control Frame (196),Perpendicularity,22.079,0.5,44.158



465 rows kept, 15 parts, 31 characteristics


### One callout, several features

Before any grouping: a `CharacteristicNominal` is the callout on the drawing,
and a `CharacteristicItem` is one place it was applied. Four holes from one hole
callout are four items against one nominal, sharing a name. `qifkit.label_items` numbers them so they stay apart.

In [5]:
clean = qifkit.label_items(clean)

items_per_name = clean.groupby('name').characteristic.nunique()
print(f'{clean.name.nunique()} callouts -> {clean.label.nunique()} individual characteristics')
print('callouts applied to more than one feature:')
print(items_per_name[items_per_name > 1].to_string())

22 callouts -> 31 individual characteristics
callouts applied to more than one feature:
name
Feature Control Frame (189)    4
Linear Dimension (114)         2
Linear Dimension (133)         2
Linear Dimension (212)         2
Radial Dimension (188)         4


## One part, as an inspector would see it

The first-article view: every characteristic, what it should be, what it was, and
how much of its tolerance it used up.

In [6]:
PART = 'SN#1'

fai = (clean[clean.serial == PART]
       .assign(percent_of_band=lambda d: (d.fraction * 100).round(1))
       .loc[:, ['label', 'type', 'nominal', 'low', 'high', 'value', 'percent_of_band', 'status_from_file']]
       .sort_values('percent_of_band', key=abs, ascending=False)
       .reset_index(drop=True))


def inspection(frame):
    # Display only: a geometric tolerance has no nominal, and a missing value is
    # clearer as a dash than as NaN.
    return (frame.rename(columns={'low': 'lower tol', 'high': 'upper tol',
                                  'percent_of_band': '% of band',
                                  'status_from_file': 'status from file'})
            .style.format({'nominal': '{:.4f}', 'lower tol': '{:+.3f}', 'upper tol': '{:+.3f}',
                           'value': '{:.4f}', '% of band': '{:+.1f}'}, na_rep='—')
            .hide(axis='index'))


print(f'{PART}: {len(fai)} characteristics, '
      f'{(fai.percent_of_band.abs() > 100).sum()} out of tolerance')
inspection(fai)

SN#1: 31 characteristics, 6 out of tolerance


label,type,nominal,lower tol,upper tol,value,% of band,status from file
Radial Dimension (190),Diameter,44.0000,-0.050,+0.150,43.8089,-382.3,FAIL
Radial Dimension (203),Diameter,44.0000,-0.050,+0.150,43.8131,-373.9,FAIL
Feature Control Frame (189) #4,Position,—,+0.000,+0.250,0.4162,+166.5,FAIL
Feature Control Frame (201),SurfaceProfile,—,+0.000,+0.250,0.3970,+158.8,UNDEFINED
Feature Control Frame (189) #3,Position,—,+0.000,+0.250,0.3573,+142.9,FAIL
Linear Dimension (194),Width,40.9878,+0.000,+0.350,40.8585,-137.0,FAIL
Feature Control Frame (200),SurfaceProfile,—,+0.000,+1.000,0.9380,+93.8,UNDEFINED
Linear Dimension (133) #1,Width,10.0000,-0.380,+0.380,9.7029,-78.2,PASS
Linear Dimension (133) #2,Width,10.0000,-0.380,+0.380,9.7192,-73.9,PASS
Linear Dimension (212) #2,DistanceBetween,6.3500,-0.050,+0.250,6.3205,-58.9,PASS


In [7]:
# Two views: characteristics out of tolerance, and those still passing but
# above 90% of the band, where the next failures are likely to come from.
failing = fai[fai.percent_of_band.abs() > 100]
marginal = fai[(fai.percent_of_band.abs() > 90) & (fai.percent_of_band.abs() <= 100)]

print(f'out of tolerance: {len(failing)}')
display(inspection(failing))
print(f'over 90% of band but passing: {len(marginal)}')
display(inspection(marginal))

out of tolerance: 6


label,type,nominal,lower tol,upper tol,value,% of band,status from file
Radial Dimension (190),Diameter,44.0000,-0.050,+0.150,43.8089,-382.3,FAIL
Radial Dimension (203),Diameter,44.0000,-0.050,+0.150,43.8131,-373.9,FAIL
Feature Control Frame (189) #4,Position,—,+0.000,+0.250,0.4162,+166.5,FAIL
Feature Control Frame (201),SurfaceProfile,—,+0.000,+0.250,0.3970,+158.8,UNDEFINED
Feature Control Frame (189) #3,Position,—,+0.000,+0.250,0.3573,+142.9,FAIL
Linear Dimension (194),Width,40.9878,+0.000,+0.350,40.8585,-137.0,FAIL


over 90% of band but passing: 1


label,type,nominal,lower tol,upper tol,value,% of band,status from file
Feature Control Frame (200),SurfaceProfile,—,+0.000,+1.000,0.9380,+93.8,UNDEFINED


## Next

[`03-across-parts`](03-across-parts.ipynb) puts all 15 parts side by side. It starts
from this same cleaned table, in one call: `qifkit.results(doc)`.